# Train SFW-SwinCBM on Google Colab
Notebook nay clone branch `ai_core`, train tren Defactify Hugging Face streaming, luu checkpoint vao Google Drive.


## 1. Check GPU
Runtime > Change runtime type > GPU. T4 la cau hinh mac dinh hop ly.


In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__)
print('cuda available', torch.cuda.is_available())


## 2. Mount Google Drive
Drive dung de luu checkpoint/output va cache nhe cho Hugging Face.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Clone repo branch ai_core
Cell nay dung Python thuan de tranh loi current directory bi xoa khi clone lai repo.


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/chtr302/ai_generated_image_detection.git'
BRANCH = 'ai_core'
REPO_DIR = Path('/content/ai_generated_image_detection')

os.chdir('/content')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run([
    'git', 'clone',
    '--single-branch',
    '--branch', BRANCH,
    REPO_URL,
    str(REPO_DIR),
], check=True)
os.chdir(REPO_DIR)
current_branch = subprocess.check_output(['git', 'branch', '--show-current'], text=True).strip()
if current_branch != BRANCH:
    raise RuntimeError(f'Expected branch {BRANCH}, got {current_branch}')
print('cwd:', Path.cwd())
print('branch:', current_branch)
subprocess.run(['git', 'log', '-1', '--oneline'], check=True)


## 4. Install dependencies
Colab co san torch/torchvision. Cell nay cai them datasets va Pillow neu thieu.


In [ ]:
import os
from pathlib import Path

REPO_DIR = Path('/content/ai_generated_image_detection')
if not REPO_DIR.exists():
    raise FileNotFoundError(f'Repo chua clone thanh cong: {REPO_DIR}')
os.chdir(REPO_DIR)
print('cwd:', Path.cwd())

!python -m pip install -q datasets pillow


## 5. Verify repo version
Kiem tra code tren Colab da co cac tham so train moi hay chua. Neu fail, can push branch `ai_core` len GitHub roi restart runtime.


In [ ]:
import subprocess
import sys

help_result = subprocess.run(
    [sys.executable, '-m', 'src.model.train', '--help'],
    check=True,
    capture_output=True,
    text=True,
)
required_args = ['--hf-dataset', '--hf-shuffle-buffer', '--max-train-steps', '--max-val-steps', '--log-every']
missing = [arg for arg in required_args if arg not in help_result.stdout]
if missing:
    subprocess.run(['git', 'branch', '--show-current'], check=False)
    subprocess.run(['git', 'log', '-1', '--oneline'], check=False)
    raise RuntimeError('Repo tren Colab dang la code cu, thieu args: ' + ', '.join(missing))
print('OK: train.py supports Colab streaming args')
subprocess.run(['git', 'log', '-1', '--oneline'], check=True)


## 6. Set data and output paths
Mac dinh dung Defactify tren Hugging Face. Neu muon dung data local tren Drive, gan `DATA_ROOT = Path('/content/drive/MyDrive/ai_data')`.


In [ ]:
from pathlib import Path

HF_DATASET = 'Rajarshi-Roy-research/Defactify_Image_Dataset'
HF_CACHE_DIR = Path('/content/drive/MyDrive/hf_cache')
HF_SHUFFLE_BUFFER = 10000
HF_NO_STREAMING = False

DATA_ROOT = None
OUTPUT_DIR = Path('/content/drive/MyDrive/ai_detector_outputs')

HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def build_data_arg():
    if DATA_ROOT is not None:
        if not Path(DATA_ROOT).exists():
            raise FileNotFoundError(f'DATA_ROOT khong ton tai: {DATA_ROOT}')
        return f'--data-root {DATA_ROOT}'

    args = f'--hf-dataset {HF_DATASET} --hf-cache-dir {HF_CACHE_DIR} --hf-shuffle-buffer {HF_SHUFFLE_BUFFER}'
    if HF_NO_STREAMING:
        args += ' --hf-no-streaming'
    return args

print('data args:', build_data_arg())
print('OUTPUT_DIR:', OUTPUT_DIR)


## 7. Quick tests
Chay unit test de bat loi import/data API truoc khi train.


In [ ]:
!PYTHONDONTWRITEBYTECODE=1 python -B -m unittest discover -s src/tests -p "test_*.py" -v


## 8. Ablation study

| Experiment | Accuracy | F1 | AUC | Note |
|---|---:|---:|---:|---|
| RGB baseline | TBD | TBD | TBD | pretrained RGB classifier |
| Spatial + Frequency | TBD | TBD | TBD | no wavelet |
| Spatial + Frequency + Wavelet | TBD | TBD | TBD | no CDAF/Swin |
| + CDAF | TBD | TBD | TBD | attention fusion |
| + Swin | TBD | TBD | TBD | fused backbone |
| Full model + CBM | TBD | TBD | TBD | current model |


## 9. Smoke train
Chay 10 batch dau de kiem tra streaming, forward, loss va checkpoint.


In [ ]:
SMOKE_IMAGE_SIZE = 128
SMOKE_BATCH_SIZE = 2
SMOKE_STEPS = 10

data_arg = build_data_arg()
!python -m src.model.train {data_arg} \
  --output-dir {OUTPUT_DIR} \
  --image-size {SMOKE_IMAGE_SIZE} \
  --batch-size {SMOKE_BATCH_SIZE} \
  --epochs 1 \
  --grad-accum 1 \
  --nec 10 \
  --amp none \
  --num-workers 0 \
  --max-train-steps {SMOKE_STEPS} \
  --max-val-steps 20 \
  --log-every 5


## 10. Debug train
Dung cell nay truoc de xem model co hoc dung huong khong. Neu `predicted_ai_rate` gan 1.0 va `specificity_real` gan 0.0 thi dung train full de sua pipeline/model.


In [ ]:
IMAGE_SIZE = 224
BATCH_SIZE = 8
EPOCHS = 3
GRAD_ACCUM = 1
NEC = 10
AMP = 'fp16'
NUM_WORKERS = 0
MAX_TRAIN_STEPS = 200
MAX_VAL_STEPS = 100
LOG_EVERY = 50

data_arg = build_data_arg()
!python -m src.model.train {data_arg} \
  --output-dir {OUTPUT_DIR} \
  --image-size {IMAGE_SIZE} \
  --batch-size {BATCH_SIZE} \
  --epochs {EPOCHS} \
  --grad-accum {GRAD_ACCUM} \
  --nec {NEC} \
  --amp {AMP} \
  --num-workers {NUM_WORKERS} \
  --max-train-steps {MAX_TRAIN_STEPS} \
  --max-val-steps {MAX_VAL_STEPS} \
  --log-every {LOG_EVERY}


## 11. Practical train
Dung sau khi debug train on. Cau hinh nay can bang thoi gian Colab va luong data.


In [ ]:
IMAGE_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 10
GRAD_ACCUM = 2
NEC = 10
AMP = 'fp16'
NUM_WORKERS = 0
MAX_TRAIN_STEPS = 1000
MAX_VAL_STEPS = 250
LOG_EVERY = 100

data_arg = build_data_arg()
!python -m src.model.train {data_arg} \
  --output-dir {OUTPUT_DIR} \
  --image-size {IMAGE_SIZE} \
  --batch-size {BATCH_SIZE} \
  --epochs {EPOCHS} \
  --grad-accum {GRAD_ACCUM} \
  --nec {NEC} \
  --amp {AMP} \
  --num-workers {NUM_WORKERS} \
  --max-train-steps {MAX_TRAIN_STEPS} \
  --max-val-steps {MAX_VAL_STEPS} \
  --log-every {LOG_EVERY}


## 12. Resume training
Dung khi Colab bi ngat runtime.


In [ ]:
RESUME = OUTPUT_DIR / 'last.pt'
if not RESUME.exists():
    raise FileNotFoundError(f'Chua co checkpoint de resume: {RESUME}. Hay train thanh cong truoc.')

data_arg = build_data_arg()
!python -m src.model.train {data_arg} \
  --output-dir {OUTPUT_DIR} \
  --image-size 256 \
  --batch-size 8 \
  --epochs 10 \
  --grad-accum 2 \
  --nec 10 \
  --amp fp16 \
  --num-workers 0 \
  --max-train-steps 1000 \
  --max-val-steps 250 \
  --log-every 100 \
  --resume {RESUME}


## 13. Inference sample
Doi `IMAGE_PATH` sang anh ban muon test.


In [ ]:
IMAGE_PATH = Path('/content/drive/MyDrive/sample.jpg')
CHECKPOINT = OUTPUT_DIR / 'best.pt'
if not IMAGE_PATH.exists():
    raise FileNotFoundError(f'Khong tim thay anh test: {IMAGE_PATH}')
if not CHECKPOINT.exists():
    raise FileNotFoundError(f'Khong tim thay checkpoint best.pt: {CHECKPOINT}. Hay train thanh cong truoc.')
!python -m src.model.inference --image {IMAGE_PATH} --checkpoint {CHECKPOINT} --image-size 256 --nec 10
